# schedule_dashboard_manifest_default_v1

目标：验证 `src.schedule` 的 model dashboard refresh 默认会执行 analysis manifest preflight，并在 legacy sidecar card 上自动 backfill manifest，然后再写 dashboard export。

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate))
        break

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = REPO_ROOT / 'research/schedule_dashboard_manifest_default_v1'
MATRIX_CARD = CARD / 'matrix_card'
LOCK_DIR = CARD / 'locks'
for path in (MATRIX_CARD, LOCK_DIR):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


In [2]:
def legacy_matrix_runner(**kwargs):
    card_dir = Path(kwargs['card_dir'])
    card_dir.mkdir(parents=True, exist_ok=True)
    (card_dir / 'analysis_summary.json').write_text(
        json.dumps(
            {
                'schema_version': 'analysis.report.v1',
                'status': 'legacy_default_backfill_completed',
                'start': '2026-01-01',
                'end': '2026-06-09',
                'model_count': 1,
                'note': '中文 review：schedule 默认 backfill manifest。',
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )
    pd.DataFrame(
        [{'rank': 1, 'model': 'ridge_mse', 'return_pct': 2.5, 'relative_return_pct': 1.0, 'stability_score': 3.0}]
    ).to_csv(card_dir / 'model_comparison.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 2.5}]).to_csv(
        card_dir / 'regime_metrics.csv', index=False
    )
    pd.DataFrame([{'model': 'ridge_mse', 'window': 'test', 'return_pct': 2.5}]).to_csv(
        card_dir / 'window_metrics.csv', index=False
    )
    pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(card_dir / 'predictions.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'nav': 1.0}]).to_csv(card_dir / 'daily_backtest.csv', index=False)
    return {'variant_count': 1, 'status': 'feature_set_matrix_completed'}


In [3]:
config = ModelDashboardPipelineConfig(
    name='schedule_dashboard_manifest_default_v1',
    card_dir=MATRIX_CARD,
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse',),
)

result = run_model_dashboard_refresh_once(
    config,
    lock_dir=LOCK_DIR,
    matrix_runner=legacy_matrix_runner,
)
manifest = json.loads((MATRIX_CARD / 'analysis_manifest.json').read_text(encoding='utf-8'))
dashboard = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))

assert result.status == 'success'
assert result.payload['analysis_manifest_preflight_status'] == 'backfilled'
assert result.payload['analysis_manifest_preflight_action'] == 'backfilled'
assert result.payload['dashboard_manifest_kind'] == 'analysis_manifest'
assert dashboard['manifest_source']['kind'] == 'analysis_manifest'
assert manifest['dataset']['name'] == 'schedule_dashboard_refresh'
assert manifest['dataset']['calendar_path'] == 'data/qlib/soxl/calendars/day.txt'
assert manifest['dataset']['universe'] == ['SOXL']
assert 'predictions' not in manifest['paths']
assert 'daily_backtest' not in manifest['paths']
result.payload


{'matrix_card': '/Users/rziiiii/dev/workspace/research/schedule_dashboard_manifest_default_v1/matrix_card',
 'dashboard_export': '/Users/rziiiii/dev/workspace/research/schedule_dashboard_manifest_default_v1/dashboard_export.json',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_artifact_id': 'matrix_card',
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_manifest_path': '/Users/rziiiii/dev/workspace/research/schedule_dashboard_manifest_default_v1/matrix_card/analysis_manifest.json',
 'dashboard_manifest_hash': '2283d3015b1061d8b795960a01b5988279cfb12863d64c4f06866f094fdda2df',
 'dashboard_manifest_producer': 'src.analysis.manifest_backfill',
 'dashboard_page': '',
 'dashboard_page_schema_version': '',
 'dashboard_view_model_schema_version': '',
 'dashboard_page_model_card_count': 0,
 'top_model': 'ridge_mse',
 'attribution_card': '',
 'attribution_status': 'not_provided',
 'matrix_status': 'feature_set_matrix_completed',
 'variant_count': 1,
 'backtest

In [4]:
summary = {
    'schema_version': 'research.schedule_dashboard_manifest_default_v1.summary.v1',
    'status': 'completed',
    'artifact_id': 'schedule_dashboard_manifest_default_v1',
    'job_status': result.status,
    'preflight_status': result.payload['analysis_manifest_preflight_status'],
    'preflight_action': result.payload['analysis_manifest_preflight_action'],
    'dashboard_manifest_kind': result.payload['dashboard_manifest_kind'],
    'dashboard_manifest_hash': result.payload['dashboard_manifest_hash'],
    'manifest_dataset_name': manifest['dataset']['name'],
    'manifest_calendar_path': manifest['dataset']['calendar_path'],
    'manifest_universe': manifest['dataset']['universe'],
    'private_paths_exposed_in_manifest': sorted(
        key for key in manifest['paths'] if key in {'predictions', 'daily_backtest', 'adapter_predictions'}
    ),
    'dashboard_top_model': dashboard['top_models'][0]['model'],
    'conclusion': '中文结论：schedule dashboard refresh 默认 backfill legacy sidecar 为 analysis_manifest，dashboard export 不再走 fixed_sidecars。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.schedule_dashboard_manifest_default_v1.summary.v1',
 'status': 'completed',
 'artifact_id': 'schedule_dashboard_manifest_default_v1',
 'job_status': 'success',
 'preflight_status': 'backfilled',
 'preflight_action': 'backfilled',
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_manifest_hash': '2283d3015b1061d8b795960a01b5988279cfb12863d64c4f06866f094fdda2df',
 'manifest_dataset_name': 'schedule_dashboard_refresh',
 'manifest_calendar_path': 'data/qlib/soxl/calendars/day.txt',
 'manifest_universe': ['SOXL'],
 'private_paths_exposed_in_manifest': [],
 'dashboard_top_model': 'ridge_mse',
 'conclusion': '中文结论：schedule dashboard refresh 默认 backfill legacy sidecar 为 analysis_manifest，dashboard export 不再走 fixed_sidecars。'}

## 中文结论

- 做了什么：验证 `ModelDashboardPipelineConfig` 默认打开 analysis manifest preflight 和 auto backfill，schedule refresh 会先把 legacy sidecar card 转成 `analysis_manifest.json`，再写 dashboard export。
- 核心指标：`preflight_status=backfilled`、`dashboard_manifest_kind=analysis_manifest`、manifest dataset 为 `schedule_dashboard_refresh`、calendar 为 `data/qlib/soxl/calendars/day.txt`。
- regime 差异：示例只使用 `bull` regime 的最小卡片，目标是验证 schedule/dashboard artifact 边界，不重新评估真实 regime 表现。
- 是否晋升：不晋升交易策略；晋升 schedule 默认契约，减少 dashboard 正式路径绕过 manifest 的机会。
- 风险：如果 matrix runner 没有产出任何可 backfill 的 analysis sidecar，preflight 会跳过，后续 dashboard export 会按 manifest requirement 失败；这是预期的 fail-closed 行为。
- 下一步：继续把无 manifest 的历史 research card 批量 backfill，或者让生产 registry/health 检查要求 dashboard export 必须带 manifest hash。